# RGB2Point sanity test: does it work on what it was trained on?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Md-Mosabbir/localpoint3d/blob/main/notebooks/test/rgb2point_sanity_test.ipynb)

**Why this notebook?** On our Bangladeshi objects, RGB2Point gave chair-like clouds (journal 06). Before blaming our objects, we check the model itself:
give it photos of things it **was** trained on and see whether the clouds look right.

The released checkpoint (`pc1024_three.pth`) was trained on ShapeNet **airplanes, chairs and cars**. So:
- an airplane photo should give an airplane-shaped cloud (wings, body, tail),
- if even that fails, the problem is the model or web photos in general, not our objects.

**Steps:** settings → install → load model → the authors' own example picture (baseline) → download a few photos → run the model → look at the clouds.

Runs on a fresh Colab runtime, top to bottom. A GPU is nice but not needed: one photo takes about 1 s on CPU.

## ⚙️ Settings

The only cell you should need to change.

- `SEARCHES`: class name → search phrase. Default is just airplanes; uncomment chair / car to test those too.
  "white background" helps because ShapeNet training pictures are renders on a plain background.
- `PHOTOS_PER_CLASS`: how many photos to download per class.
- `SEED`: makes the downloads pick the same results each run (as far as the search engine allows).

> 🧠 A **dict** (`{key: value}`) maps one thing to another, here a class name to its search phrase.

In [ ]:
SEARCHES = {
    "airplane": "passenger airplane side view white background",
    # "chair": "wooden chair white background",
    # "car": "sedan car side view white background",
}
PHOTOS_PER_CLASS = 6
SEED = 0

WORK_DIR = "/content/rgb2point_test"   # photos and clouds go here (lost when the runtime ends)

## 1. Install packages

- `timm`: the image model (ViT) RGB2Point uses inside. Pinned to the version that loads the checkpoint.
- `gdown`: downloads the checkpoint from Google Drive.
- `ddgs`, `icrawler`: DuckDuckGo and Bing image search (Google blocks scripts).
- `plotly`: interactive 3D plot you can rotate.

In [ ]:
%pip install "timm==1.0.15" "gdown>=5.2.0" "ddgs>=9.0.0" "icrawler>=0.6.10" "plotly>=5.18.0" scipy pillow matplotlib

## 2. Get RGB2Point's code and checkpoint

Clones the authors' repo and downloads their checkpoint (about 670 MB). Both are skipped if already there, so running this cell twice is fine.

One small fix: the repo asks `timm` to download ImageNet weights for its ViT. We switch that off (`pretrained=False`)
because the checkpoint already contains the ViT weights it was trained with.

> 🧠 `re.subn` replaces text with a pattern and tells you how many replacements it made; we use that count to check the fix worked.

In [ ]:
import re
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/RGB2point")
CHECKPOINT_PATH = Path("/content/pc1024_three.pth")
CHECKPOINT_ID = "1Z5luy_833YV6NGiKjGhfsfEUyaQkgua1"   # Google Drive id from the authors' README

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/JaeLee18/RGB2point.git", str(REPO_DIR)], check=True)

model_file = REPO_DIR / "model.py"
source = model_file.read_text()
source, n_fixed = re.subn(r'("vit_base_patch16_224"\s*,\s*pretrained\s*=\s*)True', r"\1False", source, count=1)
already_fixed = re.search(r'"vit_base_patch16_224"\s*,\s*pretrained\s*=\s*False', source)
if n_fixed != 1 and not already_fixed:
    raise RuntimeError("model.py changed upstream: could not switch off pretrained=True. Look at the file.")
model_file.write_text(source)

if not CHECKPOINT_PATH.exists() or CHECKPOINT_PATH.stat().st_size < 1_000_000:
    import gdown
    gdown.download(id=CHECKPOINT_ID, output=str(CHECKPOINT_PATH))

print("repo:      ", REPO_DIR)
print("checkpoint:", CHECKPOINT_PATH, f"{CHECKPOINT_PATH.stat().st_size / 2**20:.0f} MB")

## 3. Load the model

Builds the network exactly like the authors' `inference.py` (1 view in, 1,024 points out) and fills it with the checkpoint's weights.
`model.eval()` switches off training-only behaviour (dropout), so the same photo always gives the same cloud.

In [ ]:
import sys
import torch

sys.path.insert(0, str(REPO_DIR))
from model import PointCloudNet

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

model = PointCloudNet(num_views=1, point_cloud_size=1024, num_heads=4, dim_feedforward=2048)
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
model.load_state_dict(checkpoint["model"])
model = model.to(DEVICE).eval()
print("RGB2Point loaded")

## 4. Helper functions

- `photo_to_cloud`: one photo → 1,024 × 3 numbers (x, y, z). Same preprocessing as the authors:
  resize to 224 × 224, scale colours to 0–1, normalise with ImageNet's mean and spread.
- `download_photos`: searches DuckDuckGo first, Bing if that gives too few, and keeps only files that open as real images.

> 🧠 `@torch.no_grad()` tells PyTorch we only predict, not train, so it doesn't keep the extra memory training needs.

In [ ]:
import random
import numpy as np
import requests
from io import BytesIO
from PIL import Image, ImageOps
from torchvision import transforms

TO_INPUT = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])


@torch.no_grad()
def photo_to_cloud(path):
    # 1 photo -> (1024, 3) numpy array of xyz points.
    image = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    x = TO_INPUT(image).reshape(1, 1, 3, 224, 224).to(DEVICE)   # batch of 1, 1 view
    return model(x)[0].cpu().numpy()


def save_if_image(data, path, min_side=150):
    # Keep the download only if it really is a picture and not tiny.
    try:
        image = Image.open(BytesIO(data)).convert("RGB")
    except Exception:
        return False
    if min(image.size) < min_side:
        return False
    image.save(path, quality=92)
    return True


def download_photos(name, phrase, how_many, out_dir):
    out = Path(out_dir) / name
    out.mkdir(parents=True, exist_ok=True)
    have = sorted(out.glob("*.jpg"))
    if len(have) >= how_many:          # already downloaded in this session
        return have[:how_many]

    # a) DuckDuckGo: gives a list of image links; we download them one by one.
    try:
        from ddgs import DDGS
        results = DDGS().images(phrase, max_results=how_many * 4)
    except Exception as error:
        print("  DuckDuckGo failed:", error)
        results = []
    random.Random(SEED).shuffle(results)
    for r in results:
        if len(have) >= how_many:
            break
        try:
            data = requests.get(r["image"], timeout=10, headers={"User-Agent": "Mozilla/5.0"}).content
        except Exception:
            continue
        path = out / f"{name}_{len(have) + 1:02d}.jpg"
        if save_if_image(data, path):
            have.append(path)

    # b) Bing tops up if DuckDuckGo gave too few.
    if len(have) < how_many:
        from icrawler.builtin import BingImageCrawler
        tmp = out / "_bing"
        BingImageCrawler(storage={"root_dir": str(tmp)}).crawl(phrase, max_num=how_many * 2)
        for f in sorted(tmp.glob("*")):
            if len(have) >= how_many:
                break
            path = out / f"{name}_{len(have) + 1:02d}.jpg"
            if save_if_image(f.read_bytes(), path):
                have.append(path)
    return have

print("helpers ready")

## 5. Baseline: the authors' own example picture

The repo ships one demo picture, `img/1013.jpg` (a chair in a room), used by their `inference.py`.
Whatever this gives is the **best case** for web-style photos: compare every other cloud in this notebook with it.

In [ ]:
import matplotlib.pyplot as plt

VIEWS = [(0, 2, "x–z"), (0, 1, "x–y"), (2, 1, "z–y")]   # which two axes each panel shows

def show_photo_and_cloud(path, cloud):
    # Photo, then the cloud's flat "shadow" along each pair of axes.
    fig, axes = plt.subplots(1, 4, figsize=(12, 3))
    axes[0].imshow(Image.open(path))
    axes[0].set_title(Path(path).name, fontsize=8)
    for ax, (a, b, label) in zip(axes[1:], VIEWS):
        ax.scatter(cloud[:, a], cloud[:, b], s=1, color="#2a78d6")
        ax.set_aspect("equal")
        ax.set_title(label, fontsize=8)
    for ax in axes:
        ax.axis("off")
    plt.show()

example = REPO_DIR / "img" / "1013.jpg"
show_photo_and_cloud(example, photo_to_cloud(example))

## 6. Download the photos

Prints how many photos each class got. Web search can be rate-limited; if a class gets 0, wait a minute and run the cell again,
or put your own photos in `WORK_DIR/<class>/` (any `.jpg`), they are used as they are.

In [ ]:
photos = {}
for name, phrase in SEARCHES.items():
    photos[name] = download_photos(name, phrase, PHOTOS_PER_CLASS, WORK_DIR)
    print(f"{name:10s} {len(photos[name])} photos   ({phrase!r})")

## 7. Look at the photos first

Check they really show the class. A wrong photo (a toy, a cartoon, a cockpit) isn't a fair test. Stock-photo watermarks (big logos across the picture) also disturb the model, so drop those too.
To drop one, delete its file in the Colab file browser (folder icon on the left) and re-run section 6, which tops up the missing ones.

In [ ]:
import matplotlib.pyplot as plt

for name, paths in photos.items():
    fig, axes = plt.subplots(1, len(paths), figsize=(2.6 * len(paths), 2.6), squeeze=False)
    for ax, path in zip(axes[0], paths):
        ax.imshow(Image.open(path))
        ax.set_title(path.name, fontsize=8)
        ax.axis("off")
    fig.suptitle(name)
    plt.show()

## 8. Run RGB2Point on every photo

Each photo → one cloud of 1,024 points, kept in `clouds` (class → list of arrays).

> 🧠 `.shape` of a numpy array tells its size; `(1024, 3)` means 1,024 rows (points) × 3 columns (x, y, z).

In [ ]:
clouds = {name: [photo_to_cloud(p) for p in paths] for name, paths in photos.items()}
for name, cs in clouds.items():
    print(f"{name:10s} {len(cs)} clouds, each {cs[0].shape}")

## 9. Photo next to its cloud, from three sides

Each row: the photo, then the cloud seen along each axis (same plot as the baseline in section 5).
**What to look for with airplanes:** a long body, two wide wings across it, and a tail.
If you see that, RGB2Point works on its own training classes and the chair-like shapes of our objects come from them being outside its training classes.

In [ ]:
for name in clouds:
    for path, cloud in zip(photos[name], clouds[name]):
        show_photo_and_cloud(path, cloud)

## 10. Rotate one cloud yourself

Pick a class and photo number below; drag to rotate, scroll to zoom.

In [ ]:
import plotly.graph_objects as go

SHOW_CLASS, SHOW_NUMBER = list(clouds)[0], 1     # e.g. "airplane", 1 = first photo

c = clouds[SHOW_CLASS][SHOW_NUMBER - 1]
fig = go.Figure(go.Scatter3d(x=c[:, 0], y=c[:, 1], z=c[:, 2], mode="markers", marker=dict(size=2)))
fig.update_layout(title=f"{SHOW_CLASS} #{SHOW_NUMBER}", scene=dict(aspectmode="data"), height=600, margin=dict(l=0, r=0, t=40, b=0))
fig.show()

## 11. (Optional) Save the clouds as PLY

Writes one `.ply` per photo (xyz only, no colour) into `WORK_DIR/clouds/` and zips them, so you can open them in MeshLab / CloudCompare.
Download the zip from the Colab file browser.

> 🧠 A PLY file is plain text: a short header saying how many points there are, then one `x y z` line per point.

In [ ]:
import shutil

ply_dir = Path(WORK_DIR) / "clouds"
ply_dir.mkdir(parents=True, exist_ok=True)
for name in clouds:
    for path, cloud in zip(photos[name], clouds[name]):
        header = f"ply\nformat ascii 1.0\nelement vertex {len(cloud)}\nproperty float x\nproperty float y\nproperty float z\nend_header"
        np.savetxt(ply_dir / f"{path.stem}.ply", cloud, fmt="%.6f", header=header, comments="")
zip_path = shutil.make_archive(str(Path(WORK_DIR) / "rgb2point_test_clouds"), "zip", ply_dir)
print("saved", len(list(ply_dir.glob("*.ply"))), "PLY files →", zip_path)

## What the result means

Compare three things: the baseline (section 5), the airplane clouds, and our objects' clouds (journal 06).

- **Airplanes clearly airplane-shaped, ours chair-like:** the model works on its training classes, and our objects are simply outside them.
- **Airplanes as vague as ours:** the released checkpoint gives only rough shapes on real photos, whatever the object.
  It was trained on clean ShapeNet renders (one grey 3D model, plain background), and web photos are far from that.
- **Baseline vague too:** then even the authors' own demo picture doesn't give a crisp cloud, so don't expect more from any photo.

First laptop run (2026-11-22, 6 airplane photos from DuckDuckGo): the baseline and the airplanes both gave a dense column plus scattered points,
not a clear chair or airplane outline. So the vagueness is the checkpoint on real photos, not something about our objects.

Write what you saw in `journal/06_rgb2point_web_images.md` with the date.